# 02. Run Replogle Inference with the Official Checkpoint

This notebook runs the PerturbDiff sampling entrypoint on Replogle. Given control cell sets and perturbation conditions, the model generates predicted perturbed cell sets.

The first run uses the official checkpoint rather than training from scratch. `sampling.num_sampled_batches=1` keeps the run short and verifies that the data, checkpoint, and generation code are wired correctly.

In [ ]:
from pathlib import Path
import shlex
import subprocess

# Set this to the PerturbDiff repository root.
# If the notebook server is launched from the repo root, the default is already correct.
PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "src" / "apps" / "run" / "rawdata_diffusion_sampling.py").exists(), \
    "Set PROJECT_ROOT to the PerturbDiff repository root before continuing."
DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
CKPT_PATH = PROJECT_ROOT / "checkpoints" / DATASET_NAME / "from_scratch_replogle.ckpt"
RUN_DIR = PROJECT_ROOT / "runs" / "replogle_sampling_demo"
WANDB_DIR = PROJECT_ROOT / "wandb"

print("PERTURB_ROOT:", PERTURB_ROOT)
print("CKPT_PATH:", CKPT_PATH)
print("RUN_DIR:", RUN_DIR)

assert PERTURB_ROOT.exists(), "Run 00_download_data.ipynb first."
assert CKPT_PATH.exists(), "Run 00_download_data.ipynb first."

## 1. Build the Hydra Command

The sampling entrypoint is `src/apps/run/rawdata_diffusion_sampling.py`. The overrides below follow the Replogle settings in the repository and reduce the number of sampled batches for a short validation run.

In [ ]:
cmd = [
    "python", "./src/apps/run/rawdata_diffusion_sampling.py",
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    f"model_checkpoint_path={CKPT_PATH}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
    "cov_encoding=trixie_onehot",
    "cov_encoding.batch_encoding=onehot",
    "cov_encoding.celltype_encoding=llm",
    "cov_encoding.replogle_gene_encoding=genept",
    "model.p_drop_control=0",
    "data.keep_control_cell=false",
    "sampling.use_ddim=true",
    "sampling.num_sampled_batches=1",
    "trainer.devices=[0]",
    "trainer.use_distributed_sampler=false",
    "lightning.logger._target_=pytorch_lightning.loggers.logger.DummyLogger",
    "~lightning.logger.project",
    "~lightning.logger.save_dir",
    "~lightning.logger.name",
]

print(" ".join(shlex.quote(x) for x in cmd))

## 2. Run Inference

The first run checks three things:
- Required data files and embeddings are available.
- The checkpoint loads successfully.
- The output directory receives generated result files.

In [ ]:
# Run inference. If GPU memory is limited, reduce optimization.micro_batch_size and data.use_cell_set.
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True)
print("return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Sampling command failed. Check the terminal output above for the first real error.")

## 3. Inspect Output Files

PerturbDiff writes generated results under the run directory. File names may vary by configuration, so this cell scans common output formats recursively.

In [ ]:
output_files = []
for pattern in ["*.npz", "*.h5", "*.h5ad", "*.csv", "*.pkl"]:
    output_files.extend(RUN_DIR.rglob(pattern))

if not output_files:
    print("No common output files found under", RUN_DIR)
else:
    for path in sorted(output_files)[:20]:
        size_mb = path.stat().st_size / 1024**2
        print(f"{size_mb:8.2f} MB  {path}")

## 4. Read `.npz` Results and Inspect Shapes

If the output format is `.npz`, this cell prints each array name and shape. The final dimension typically corresponds to the 2,000 HVG expression space.

In [ ]:
npz_files = sorted(RUN_DIR.rglob("*.npz"))
if npz_files:
    import numpy as np
    npz_path = npz_files[0]
    print("Reading:", npz_path)
    data = np.load(npz_path, allow_pickle=True)
    for key in data.files:
        arr = data[key]
        print(f"{key:30s} shape={arr.shape} dtype={arr.dtype}")
else:
    print("No .npz file found. Inspect the files listed above for the configured output format.")